# 第 1 天课堂资产项目 —— 比较多类资产

作者：Febryan（[LinkedIn](https://www.linkedin.com/in/febryanmughni/)）

## 练习目标（理念）

用 **yfinance** 拉取过去约 3 年的行情（BTC、黄金期货、S&P 500），算出简单统计量，再交给 **OpenAI Chat Completions** 做资产类别对比与投资视角解读。

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 外部数据 → 文本上下文 | `analyze_asset` + `data_summary` f-string |
| Chat Completions | `client.chat.completions.create(...)` |
| `messages`（system / user） | 金融分析师人设 + 带数据的 user 问题 |
| `.env` 密钥 | `OPENAI_API_KEY` + 常见格式自检 |

## 怎么跑

1. 安装依赖：`yfinance`、`pandas`、`openai`、`python-dotenv` 等
2. 在 `.env` 里配置 `OPENAI_API_KEY`
3. 从上到下运行：先下载数据 → 校验密钥 → 统计 → 调 LLM → 展示 Markdown 分析


In [4]:
# ========== 导入：行情数据、环境变量、展示、OpenAI ==========

# 导入 yfinance 并起别名 yf：从 Yahoo Finance 拉历史行情
import yfinance as yf
# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量
from dotenv import load_dotenv
# 从本地 scraper 导入抓取函数（本 notebook 后续未直接调用，但保留原导入）
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [5]:
# ========== 下载三类资产近 3 年日频行情 ==========

# BTC-USD：比特币对美元；period="3y" 表示约三年窗口
btc = yf.download("BTC-USD", period="3y")
# ^GSPC：标普 500 指数（S&P 500）在 Yahoo 上的代码
spx = yf.download("^GSPC", period="3y")
# GC=F：黄金期货连续合约代码
gold = yf.download("GC=F", period="3y")

# 打印原始 DataFrame，便于目检列名（Close 等）与 MultiIndex 形态
print(btc)
print(spx)
print(gold)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Price              Close          High           Low          Open  \
Ticker           BTC-USD       BTC-USD       BTC-USD       BTC-USD   
Date                                                                 
2023-01-18  20688.781250  21564.501953  20541.544922  21161.050781   
2023-01-19  21086.792969  21163.011719  20685.380859  20686.746094   
2023-01-20  22676.552734  22692.357422  20919.126953  21085.373047   
2023-01-21  22777.625000  23282.347656  22511.833984  22677.427734   
2023-01-22  22720.416016  23056.730469  22387.900391  22777.986328   
...                  ...           ...           ...           ...   
2026-01-13  95321.781250  96011.625000  90941.929688  91185.335938   
2026-01-14  96929.328125  97860.601562  94583.046875  95322.906250   
2026-01-15  95551.187500  97150.171875  95103.242188  96931.289062   
2026-01-16  95525.117188  95801.890625  94259.273438  95554.101562   
2026-01-18  95019.250000  95186.789062  94854.273438  95112.398438   

Price            Vo

In [6]:
# ========== 加载 .env 并做 OpenAI API Key 基础体检 ==========

# override=True：即使环境里已有同名变量，也用 .env 覆盖（与课程常见写法一致）
load_dotenv(override=True)
# 从环境变量读取 OPENAI_API_KEY（注意名字是 OPENAI_API_KEY，不是 OPENAI_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# 下面三段 print 文案保持英文原样：与课程 troubleshooting 笔记本提示一致，勿改译

# 情况 1：完全没读到密钥
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 情况 2：读到了但不像 sk-proj- 开头的项目密钥
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 情况 3：首尾有空格/制表符（strip 后与原串不同）
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
# 情况 4：通过以上粗检
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## 一步一步：构建 LLM 财务分析代理

下面把「行情统计 → 文本上下文 → Chat Completions → Markdown 展示」拆成 6 步。这里的 **LLM** 指大语言模型（Large Language Model），不是「法学硕士」。

### 第 1 步：分析并准备财务数据

从 BTC、黄金和 SPX 的收盘价序列提取关键统计（总回报、波动、高低价区间）。

### 第 2 步：创建给 LLM 的数据摘要

把统计结果格式化成模型容易读的英文文本字符串（`data_summary`）。

### 第 3 步：设置 OpenAI 客户端

用已校验的 `api_key` 初始化 `OpenAI(...)`。

### 第 4 步：创建 system / user 提示

system 定「专业资产比较分析师」；user 嵌入 `data_summary` 并提出对比问题。

### 第 5 步：调用 LLM API

`chat.completions.create` 发送 `messages`，取出 `message.content`。

### 第 6 步：显示结果

用 `display(Markdown(...))` 在笔记本里可读地展示分析。


In [8]:
# ========== 第 1 步：从收盘价序列计算每类资产的关键统计 ==========

# 导入 pandas：判断 MultiIndex 列、做序列统计
import pandas as pd

def analyze_asset(data, name):
    """计算单资产：起止价、总回报%、最高/最低价、收盘价标准差（波动代理）。"""
    # yfinance 多标的下载时列常为 MultiIndex：(字段, 代码)
    if isinstance(data.columns, pd.MultiIndex):
        # MultiIndex：用 ('Close', name) 取出该代码的收盘价序列
        close_prices = data[('Close', name)]
    else:
        # 单层列名：直接取 'Close'
        close_prices = data['Close']
    
    # iloc[0] / iloc[-1]：窗口内第一天与最后一天收盘价
    start_price = float(close_prices.iloc[0])
    end_price = float(close_prices.iloc[-1])
    # 简单总回报（%）=(终-始)/始 * 100；未年化、未计分红
    total_return = ((end_price - start_price) / start_price) * 100
    # 区间最高 / 最低收盘价
    max_price = float(close_prices.max())
    min_price = float(close_prices.min())
    # 用收盘价标准差当粗波动指标（单位与价格相同，不是收益率波动）
    volatility = float(close_prices.std())
    
    # 返回字典：键名保持英文，后面 f-string / print 都按这些键取
    return {
        'Asset': name,
        'Start Price': start_price,
        'End Price': end_price,
        'Total Return (%)': round(total_return, 2),
        'Max Price': max_price,
        'Min Price': min_price,
        'Volatility': round(volatility, 2)
    }

# 分别分析三类资产；第二个参数须与 yfinance 列上的代码一致
btc_stats = analyze_asset(btc, 'BTC-USD')
spx_stats = analyze_asset(spx, '^GSPC')
gold_stats = analyze_asset(gold, 'GC=F')

# 人类可读的汇总打印（emoji 与英文标签保持原样）
print("📊 Asset Statistics (3 Years):")
print("\n" + "="*60)
for stats in [btc_stats, spx_stats, gold_stats]:
    print(f"\n{stats['Asset']}:")
    print(f"  Return: {stats['Total Return (%)']}%")
    print(f"  Start Price: ${stats['Start Price']:,.2f}")
    print(f"  End Price: ${stats['End Price']:,.2f}")
    print(f"  Volatility: ${stats['Volatility']:,.2f}")


📊 Asset Statistics (3 Years):


BTC-USD:
  Return: 359.28%
  Start Price: $20,688.78
  End Price: $95,019.25
  Volatility: $31,428.03

^GSPC:
  Return: 73.89%
  Start Price: $3,990.97
  End Price: $6,940.01
  Volatility: $879.78

GC=F:
  Return: 140.58%
  Start Price: $1,907.20
  End Price: $4,588.40
  Volatility: $734.06


In [9]:
# ========== 第 2 步：把统计字典格式化成将发给 LLM 的英文摘要 ==========

# f-string 多行模板：字段标签与数字格式保持英文（这是模型输入上下文，勿改译）
data_summary = f"""
3-YEAR FINANCIAL DATA (2023-2026):

BITCOIN (BTC-USD):
- Start Price: ${btc_stats['Start Price']:,.2f}
- End Price: ${btc_stats['End Price']:,.2f}
- Total Return: {btc_stats['Total Return (%)']}%
- Highest Price: ${btc_stats['Max Price']:,.2f}
- Lowest Price: ${btc_stats['Min Price']:,.2f}
- Volatility: ${btc_stats['Volatility']:,.2f}

S&P 500 (^GSPC):
- Start Price: ${spx_stats['Start Price']:,.2f}
- End Price: ${spx_stats['End Price']:,.2f}
- Total Return: {spx_stats['Total Return (%)']}%
- Highest Price: ${spx_stats['Max Price']:,.2f}
- Lowest Price: ${spx_stats['Min Price']:,.2f}
- Volatility: ${spx_stats['Volatility']:,.2f}

GOLD (GC=F):
- Start Price: ${gold_stats['Start Price']:,.2f}
- End Price: ${gold_stats['End Price']:,.2f}
- Total Return: {gold_stats['Total Return (%)']}%
- Highest Price: ${gold_stats['Max Price']:,.2f}
- Lowest Price: ${gold_stats['Min Price']:,.2f}
- Volatility: ${gold_stats['Volatility']:,.2f}
"""

# 先打印摘要，方便确认即将塞进 user prompt 的文本
print("📋 Data Summary to be sent to LLM:")
print(data_summary)


📋 Data Summary to be sent to LLM:

3-YEAR FINANCIAL DATA (2023-2026):

BITCOIN (BTC-USD):
- Start Price: $20,688.78
- End Price: $95,019.25
- Total Return: 359.28%
- Highest Price: $124,752.53
- Lowest Price: $20,187.24
- Volatility: $31,428.03

S&P 500 (^GSPC):
- Start Price: $3,990.97
- End Price: $6,940.01
- Total Return: 73.89%
- Highest Price: $6,977.27
- Lowest Price: $3,855.76
- Volatility: $879.78

GOLD (GC=F):
- Start Price: $1,907.20
- End Price: $4,588.40
- Total Return: 140.58%
- Highest Price: $4,626.30
- Lowest Price: $1,808.80
- Volatility: $734.06



In [10]:
# ========== 第 3 步：用已读取的 api_key 创建 OpenAI 客户端 ==========

# OpenAI(...)：默认打官方 API；密钥来自上格的 OPENAI_API_KEY
client = OpenAI(api_key=api_key)

# 状态提示字符串保持原样（含 emoji）
print("✅ OpenAI Client is ready to use!")


✅ OpenAI Client is ready to use!


In [11]:
# ========== 第 4 步：编写 system / user 提示（发给模型的英文保持原样）==========

# system_prompt：定「专业资产类别比较分析师」角色与输出结构要求
system_prompt = """You are a professional financial analyst specializing in asset class comparison. 
Your role is to analyze financial data and provide:
1. Clear comparison between different asset classes
2. Insights about performance, risk, and volatility
3. Recommendations based on the data
4. Context about market conditions

Use Markdown formatting for better readability with headers, bullet points, and clear sections."""

# user_prompt：嵌入 data_summary，并列出希望模型回答的四个分析角度
user_prompt = f"""Please analyze the following financial data and provide a comprehensive comparison between Bitcoin, S&P 500, and Gold:

{data_summary}

Please provide:
1. Performance comparison between the three assets
2. Risk and volatility analysis
3. Investment recommendations based on this data
4. Insights about market trends over the past 3 years

Format your response with clear sections and use Markdown formatting."""

# 确认提示已构造完成（打印文案保持原样）
print("✅ Prompts have been created!")


✅ Prompts have been created!


In [12]:
# ========== 第 5 步：组装 messages 并调用 Chat Completions ==========

# messages：system 定角色，user 带数据与具体问题
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# 进度提示：即将发请求（字符串保持原样）
print("🔄 Sending request to OpenAI...")

try:
    # chat.completions.create：非流式一次拿完整回复
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # 更快、更省的小模型；model id 勿改
        messages=messages,
        temperature=0.7  # 0.7：在创意与稳健之间折中
    )
    
    # 成功：取出助手文本
    print("✅ Response received successfully!")
    analysis_result = response.choices[0].message.content
    
except Exception as e:
    # 失败：打印错误并把结果置空，供下一格分支处理（错误文案格式保持原样）
    print(f"❌ Error: {e}")
    analysis_result = None


🔄 Sending request to OpenAI...
✅ Response received successfully!


In [13]:
# ========== 第 6 步：若有结果则用 Markdown 展示，否则提示无结果 ==========

if analysis_result:
    # 打印分隔条与英文标题（展示文案保持原样）
    print("\n" + "="*60)
    print("🤖 LLM ANALYSIS RESULTS")
    print("="*60 + "\n")
    # 把模型返回的 Markdown 渲染到笔记本输出区
    display(Markdown(analysis_result))
else:
    # 上一格失败时走这里；字符串保持英文原样
    print("No results to display.")



🤖 LLM ANALYSIS RESULTS



# Comprehensive Asset Class Comparison: Bitcoin, S&P 500, and Gold

## 1. Performance Comparison

### Total Returns
- **Bitcoin (BTC-USD)**
  - **Total Return:** 359.28%
  - **Price Movement:** From $20,688.78 to $95,019.25

- **S&P 500 (^GSPC)**
  - **Total Return:** 73.89%
  - **Price Movement:** From $3,990.97 to $6,940.01

- **Gold (GC=F)**
  - **Total Return:** 140.58%
  - **Price Movement:** From $1,907.20 to $4,588.40

### Summary of Performance
- Bitcoin has significantly outperformed both the S&P 500 and Gold, achieving a return of over 350% in just three years.
- Gold has shown respectable growth, more than doubling its value, while the S&P 500 has increased by nearly 74%.

## 2. Risk and Volatility Analysis

### Volatility
- **Bitcoin (BTC-USD)**
  - **Volatility:** $31,428.03 
  - **Implication:** Extremely high volatility; indicative of the speculative nature of crypto assets.

- **S&P 500 (^GSPC)**
  - **Volatility:** $879.78
  - **Implication:** Moderate volatility; reflects a diversified portfolio of large-cap stocks.

- **Gold (GC=F)**
  - **Volatility:** $734.06
  - **Implication:** Lower volatility compared to Bitcoin and slightly lower than the S&P 500; typically seen as a safe-haven asset.

### Risk Assessment
- **Bitcoin** presents the highest risk due to its extreme price fluctuations, making it suitable for high-risk investors looking for potentially high returns.
- **S&P 500** offers a balanced approach with moderate risk, suitable for long-term investors seeking steady growth.
- **Gold** is the least volatile and is generally favored by conservative investors looking to hedge against market uncertainty.

## 3. Investment Recommendations

### For High-Risk Tolerance
- **Invest in Bitcoin:** Given its astronomical growth, Bitcoin is appealing for those who can handle high volatility and are looking for substantial returns. 

### For Moderate Risk Tolerance
- **Invest in S&P 500:** The S&P 500 is ideal for investors looking for moderate growth with reasonable risk. It provides exposure to the broader economy and benefits from corporate earnings growth.

### For Low-Risk Tolerance
- **Invest in Gold:** Gold serves as a protective asset and is a reliable hedge against inflation and market downturns. This is suitable for conservative investors or those nearing retirement.

## 4. Insights About Market Trends Over the Past 3 Years

- **Bitcoin's Surge:** The significant increase in Bitcoin’s price reflects growing institutional adoption, a shift towards digital assets, and the increasing use of blockchain technology.

- **S&P 500 Performance:** The S&P 500's growth has been driven by a recovery from the COVID-19 pandemic, strong corporate earnings, and low-interest rates, despite facing challenges like inflation and geopolitical tensions.

- **Gold as a Safe Haven:** Gold's performance indicates its traditional role as a hedge against inflation and market uncertainty, especially during periods of high volatility in both the stock market and emerging asset classes like cryptocurrencies.

### Conclusion
The analysis showcases that each asset class has its own unique characteristics and risk profiles. Investors should align their investment choices with their risk tolerance, investment horizon, and market outlook. Diversification across these asset classes can also be a prudent strategy to balance risk and return.